# DenseGRPO 最小可运行 Demo

本 notebook 对应理论笔记 `10_denseGRPO.ipynb`，实现 DenseGRPO 的两个核心机制：

1. **Step-wise Dense Reward**：保存 SDE 去噪链的全部状态；从每个中间状态用确定性 ODE 补全到终点并评分，得到 $R_t$ 和 $\Delta R_t=R_{t+1}-R_t$。
2. **Exploration Space Calibration**：根据每个时间步奖励增量的正负平衡程度，更新逐步噪声表 $\psi(t)$。

策略更新仍是 PPO/GRPO surrogate，但优势不再是每条轨迹一个 `[G]` 标量，而是逐步优势 `[G,T]`。

参考：DenseGRPO 论文笔记、项目 `online_RL/src/online_rl/algorithms/dense_grpo/algorithm.py` 和 `trainer.py::_compute_dense_rewards`。本例用二维 latent 和小 MLP 替代图像/车辆规划模型，但保留 rollout、replay、log-prob、dense reward 和校准的数据依赖。


## 1. 张量契约与时间方向

本例令 $t=0$ 为噪声端、$t=1$ 为数据端，离散步长 $\Delta t=1/T>0$。

- 组大小 `G`：同一个条件下采样的候选数；
- latent 维度 `D=2`；
- `states [G,T+1,D]`：SDE 链状态；
- `old_log_probs [G,T]`：old policy 对已采样转移的 log-prob；
- `intermediate_rewards [G,T+1]`；
- `dense_rewards [G,T] = R[:,1:] - R[:,:-1]`；
- `advantages [G,T]`：对每个时间步独立做组内标准化。

这里的 SDE 转移简化为同方差高斯策略：

$$x_{t+\Delta t}\sim\mathcal N(x_t+\Delta t\,v_\theta(x_t,t,c),\;\psi_t^2\Delta t\,I).$$


In [ ]:
import math
from copy import deepcopy

import torch
import torch.nn as nn

torch.manual_seed(11)
device = torch.device('cpu')

G = 16                 # 同一场景的一组候选数
T = 6                  # 去噪/SDE 决策步数
D = 2                  # 教学 latent 维度
LR = 2e-3
CLIP_RANGE = 0.2

class TinyFlowPolicy(nn.Module):
    """v_theta(x,t,c): x=[B,D], t=[B], c=[B,1] -> velocity=[B,D]。"""
    def __init__(self, hidden=48):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(D + 1 + 1, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden), nn.Tanh(),
            nn.Linear(hidden, D),
        )

    def forward(self, x, t, cond):
        return self.net(torch.cat([x, t[:, None], cond], dim=-1))

def reward_fn(x_final, cond):
    """目标随条件变化；返回每个候选一个常量奖励 [B]。"""
    goal = torch.cat([1.5 + cond, -1.0 + 0.5 * cond], dim=-1)
    return -(x_final - goal).square().sum(dim=-1)

def group_normalize_per_step(values, eps=1e-6):
    """values=[G,T]，沿候选维 G 对每个 t 独立标准化。"""
    mean = values.mean(dim=0, keepdim=True)
    std = values.std(dim=0, keepdim=True, correction=0)
    return ((values - mean) / (std + eps)).detach()


In [ ]:
@torch.no_grad()
def sde_rollout(old_policy, cond, psi):
    """用冻结 old policy 收集完整随机轨迹及 rollout log-prob。"""
    batch = cond.shape[0]
    dt = 1.0 / T
    x = torch.randn(batch, D, device=cond.device)
    states = [x.clone()]
    log_probs, means, stds = [], [], []
    for step in range(T):
        t = torch.full((batch,), step / T, device=cond.device)
        velocity = old_policy(x, t, cond)
        mean = x + dt * velocity
        std = (psi[step] * math.sqrt(dt)).clamp_min(1e-4)
        dist = torch.distributions.Normal(mean, std)
        x_next = dist.sample()
        log_prob = dist.log_prob(x_next).sum(dim=-1)
        states.append(x_next.clone())
        log_probs.append(log_prob)
        means.append(mean)
        stds.append(torch.full_like(mean, std))
        x = x_next
    return {
        'states': torch.stack(states, dim=1),       # [G,T+1,D]
        'old_log_probs': torch.stack(log_probs, 1),# [G,T]
        'old_means': torch.stack(means, 1),        # [G,T,D]
        'stds': torch.stack(stds, 1),              # [G,T,D]
    }

@torch.no_grad()
def ode_finish(policy, x_start, cond, start_step, substeps=8):
    """从第 start_step 个 SDE 状态确定性积分到 t=1。"""
    x = x_start.clone()
    t0 = start_step / T
    if t0 >= 1.0:
        return x
    dt = (1.0 - t0) / substeps
    for local in range(substeps):
        t = torch.full((x.shape[0],), t0 + local * dt, device=x.device)
        x = x + dt * policy(x, t, cond)
    return x

@torch.no_grad()
def compute_dense_rewards(old_policy, rollout, cond):
    """R_j=reward(ODE-finish(x_j))，终点使用真实 SDE endpoint。"""
    states = rollout['states']
    rewards = []
    for j in range(T):
        estimated_final = ode_finish(old_policy, states[:, j], cond, j)
        rewards.append(reward_fn(estimated_final, cond))
    rewards.append(reward_fn(states[:, -1], cond))
    intermediate = torch.stack(rewards, dim=1)       # [G,T+1]
    dense = intermediate[:, 1:] - intermediate[:, :-1]  # [G,T]
    return intermediate, dense

def replay_log_probs(theta, rollout, cond, psi):
    """在已收集的 states/actions 上重算 theta log-prob，保留梯度。"""
    states = rollout['states']
    dt = 1.0 / T
    new_log_probs = []
    for step in range(T):
        x, action = states[:, step], states[:, step + 1].detach()
        t = torch.full((x.shape[0],), step / T, device=x.device)
        mean = x + dt * theta(x, t, cond)
        std = (psi[step] * math.sqrt(dt)).clamp_min(1e-4)
        new_log_probs.append(torch.distributions.Normal(mean, std).log_prob(action).sum(-1))
    return torch.stack(new_log_probs, dim=1)          # [G,T]


## 2. DenseGRPO loss 与探索校准

PPO surrogate 对每个 `(candidate,time)` 元素分别计算：

$$L=-\operatorname{mean}\left[\min(r_{g,t}A_{g,t},\operatorname{clip}(r_{g,t},1-\epsilon,1+\epsilon)A_{g,t})\right].$$

校准部分是论文思想的最小版本：若某步 $\Delta R_t$ 的正负比例接近平衡，说明探索空间仍有辨别力，增加 $\psi_t$；若几乎同号，减小 $\psi_t$。实际训练应跨多个 rollout 累积符号统计，而不是只看一次小组。


In [ ]:
def dense_grpo_loss(new_logp, old_logp, advantages, clip_range=CLIP_RANGE):
    ratio = (new_logp - old_logp).exp()
    surr1 = ratio * advantages
    surr2 = ratio.clamp(1 - clip_range, 1 + clip_range) * advantages
    loss = -torch.minimum(surr1, surr2).mean()
    clip_fraction = ((ratio.detach() - 1).abs() > clip_range).float().mean()
    return loss, ratio, clip_fraction

@torch.no_grad()
def calibrate_psi(psi, dense_rewards, balance_threshold=0.35, step_size=0.03):
    pos_fraction = (dense_rewards > 0).float().mean(dim=0)
    neg_fraction = (dense_rewards < 0).float().mean(dim=0)
    imbalance = (pos_fraction - neg_fraction).abs()
    direction = torch.where(imbalance < balance_threshold, 1.0, -1.0)
    return (psi + step_size * direction).clamp(0.08, 0.8), imbalance


In [ ]:
theta = TinyFlowPolicy().to(device)
optimizer = torch.optim.Adam(theta.parameters(), lr=LR)
psi = torch.full((T,), 0.35, device=device)
cond = torch.zeros(G, 1, device=device)  # 同一条件的一组 G 个候选

history = []
for iteration in range(12):
    # old 是本轮 rollout/replay 的冻结锚点。
    old = deepcopy(theta).eval()
    for p in old.parameters():
        p.requires_grad_(False)

    rollout = sde_rollout(old, cond, psi)
    intermediate_R, delta_R = compute_dense_rewards(old, rollout, cond)
    advantages = group_normalize_per_step(delta_R)

    new_logp = replay_log_probs(theta, rollout, cond, psi)
    loss, ratio, clip_frac = dense_grpo_loss(
        new_logp, rollout['old_log_probs'].detach(), advantages
    )
    optimizer.zero_grad()
    loss.backward()
    grad_norm = torch.nn.utils.clip_grad_norm_(theta.parameters(), 1.0)
    optimizer.step()

    # 教学中每轮校准；正式实现应按 interval 跨 rollout 累积后更新。
    psi, imbalance = calibrate_psi(psi, delta_R)
    terminal_mean = intermediate_R[:, -1].mean().item()
    history.append(terminal_mean)
    print(f'iter={iteration:02d} loss={loss.item():+.4f} grad={grad_norm.item():.3f} '
          f'R_terminal={terminal_mean:+.3f} clip={clip_frac.item():.2f} '
          f'psi={psi.tolist()}')

print('shapes:', {
    'states': tuple(rollout['states'].shape),
    'intermediate_R': tuple(intermediate_R.shape),
    'delta_R': tuple(delta_R.shape),
    'advantages': tuple(advantages.shape),
    'log_probs': tuple(new_logp.shape),
})
assert rollout['states'].shape == (G, T + 1, D)
assert delta_R.shape == advantages.shape == new_logp.shape == (G, T)
assert torch.isfinite(loss)


## 3. 与完整 RL 框架的映射

| Demo | `online_RL` |
|---|---|
| `sde_rollout` | `DrivingStage.generate_sde_with_schedule` |
| `ode_finish` | `DrivingStage.ode_denoise_and_decode` |
| `compute_dense_rewards` | `OnlineRLTrainer._compute_dense_rewards` |
| `group_normalize_per_step` | `DenseGRPO.compute_dense_advantages` |
| `dense_grpo_loss` | `DenseGRPO.compute_loss` |
| `calibrate_psi` | `DenseGRPO._calibrate_psi` |

最重要的正确性条件：中间奖励必须由中间 latent **确定性补全后**的可评分输出得到；不能直接给高噪声 latent 打终端奖励。每个时间步必须独立归一化，不能把 `[G,T]` 全部展平后做一次标准化。
